# ContractRisk — juez 12: continuación con facturación activa

Runtime CPU; no necesita T4. Usa el mismo proyecto y los secretos GEMINI_API_KEY y WANDB_API_KEY.
No repite Comparación 02 ni regenera respuestas contractuales. Conserva la única nota válida,
las ocho reservas anteriores y el techo de USD 5 TOTALES (no adicionales).

Estado de cuenta aportado: nivel 1, prepago, recarga automática desactivada. El panel muestra
COP 0,00 con advertencia de retraso; NO prueba consumo definitivo cero. La recarga COP 60.000
no aumenta nuestro presupuesto autorizado. No se usan otras aplicaciones con este proyecto
durante esta ejecución. No comprar más créditos ni borrar registros para repetir.

Mismo modelo, rúbrica, preguntas, semillas y configuración de generación; modo pago explícito.
El registro inicial debe coincidir con los ocho asientos por USD 3,6994725 estimados/reservados.
Margen local inicial: USD 1,3005275; no es saldo del proveedor. No libera reservas desconocidas.
Una petición reserva como máximo USD 0,6144 según tarifas verificadas el 19/11/2026.
Vigencia local hasta 20/11/2026 UTC; después se detiene y requiere revisar tarifas.

No hay reintentos automáticos de transporte: un 503, timeout o error de red detiene la corrida.
Un formato inválido admite una reparación, como en el protocolo anterior; una nota válida no
se repite ni ajusta. Tras el piloto aprobado continúa la regresión, siempre dentro del presupuesto.
Incluso si pasa, faltan auditar motivos y ejecutar la confirmación independiente: no valida
automáticamente al juez para calificar el producto. No garantiza disponibilidad ni juicio correcto.

Ejecutar una sola vez y en orden. Si una celda falla, saltar a las celdas de inspección y descarga;
no reiniciar la ejecución ni eliminar marcadores. Conservar ZIP y notebook con salidas.


In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import userdata

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_hydrogen_paid12_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
# Required by historical-integrity unit tests; no inference from that commit is rerun.
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin',
    '6ec8dcdf22433c632b5eb0e5539401c33b9c923d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '71912f1c5755790381a8e39c8a67801bcbf9243d'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '7afdda0212bf38ad1accd043a968db722569c618'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', 'ece58275c2a19abfb6bbd2fe4b507731232e9083'], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)
for name in ['GEMINI_API_KEY', 'WANDB_API_KEY']:
    try:
        value = userdata.get(name)
        if not value:
            raise ValueError('Empty secret')
        os.environ[name] = value
        del value
    except Exception:
        raise RuntimeError('Habilita el acceso de ESTE notebook al secreto ' + name + '. No pegues la clave.') from None

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError('Proceso detenido; conservar resultados y consultar ' + logfile)


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'final/requirements-api-judge.txt'], 'installation.log')
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('Anclas aprobadas y hashes comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUDGET_DIR = Path('/content/drive/MyDrive/ContractRisk_final_budget')
print('No borrar ni cambiar el registro de presupuesto ni sus marcadores.')
confirmation = input('Confirma facturación activa en el mismo proyecto, recarga automática desactivada, ninguna otra aplicación/usuario consumiendo esta API y techo TOTAL USD 5. Escribe PAGO_MAXIMO_5_USD: ')
assert confirmation.strip() == 'PAGO_MAXIMO_5_USD', 'Sin confirmación no se consulta la API.'
del confirmation
# Esta comprobación solo lee Drive y archivos; no consulta Google ni W&B.
from final.hydrogen_paid12 import verify_checkpoint, check_drive
cfg = json.loads((ROOT / 'final/hydrogen_calibration_config.json').read_text(encoding='utf-8'))
checkpoint = verify_checkpoint(ROOT)
check_drive(BUDGET_DIR, cfg, checkpoint)
print('Historial de ocho asientos y cinco marcadores verificado; ninguna consulta de modelo realizada en esta celda.')


In [ ]:
RESULTS = ROOT / 'hydrogen_paid12_results'
run_visible([sys.executable, '-u', 'final/run_hydrogen_paid12.py', '--output', str(RESULTS),
    '--budget-dir', str(BUDGET_DIR), '--confirm-paid-project', '--confirm-exclusive-budget-use',
    '--require-wandb'], 'hydrogen_paid12.log')


## Interpretación

La nota histórica se conserva intacta. judgements_total incluye esa nota;
judgements_executed cuenta nuevos casos y generation_http_calls cuenta peticiones,
incluida una eventual reparación de formato. No mezclar esos denominadores.

Si se detiene, conservar los resultados y no borrar registros de Drive. No repetir
la ejecución, aumentar el presupuesto ni comprar créditos para intentar de nuevo.
Pasar la regresión no certifica al juez: faltan auditar motivos y la confirmación
independiente. Cero consumo visible en el panel no prueba ausencia de coste pendiente.


In [ ]:
RESULTS = ROOT / 'hydrogen_paid12_results'
for name in ['metadata.json', 'pilot_summary.json', 'regression_summary.json', 'partial_regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for logfile in ROOT.glob('*.log'):
    shutil.copy2(logfile, RESULTS / logfile.name)
for name in ['hydrogen_calibration_config.json', 'hydrogen_budget_policy.json', 'api_budget_policy.json']:
    shutil.copy2(ROOT / 'final' / name, RESULTS / name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
(RESULTS / 'paid12_policy.json').write_text(json.dumps(__import__('final.hydrogen_paid12', fromlist=['POLICY']).POLICY, indent=2), encoding='utf-8')
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez12_pago_resultados'), 'zip', RESULTS)
files.download(package)
